# Preprocessing Technique: Outlier Detection & Removal
**Member:** [Nirukshan R / IT25101819]
**Technique:** Outlier Removal
**Dataset:** Tourism Recommendation Dataset

## Why this technique was needed
Numeric fields like `spend_amount`, `ticket_price`, and `visit_duration_hours` are exactly the
kind of self-reported/transactional fields prone to extreme outliers (e.g. data entry errors,
rare luxury spenders). Outliers can distort distance-based models (K-Means) and skew scaling
(StandardScaler), so they need to be identified and handled before modeling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("output_after_feature_selection.csv")
numeric_cols = ["ticket_price", "spend_amount"]
df[numeric_cols].describe()

,ticket_price,spend_amount
count,100000.000000,100000.000000
mean,80.783510,255.433621
std,67.183866,179.814679
min,0.000000,0.000000
25%,35.000000,98.000000
50%,70.000000,227.200000
75%,115.000000,401.090000
max,399.000000,898.220000


In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

outlier_report = {}
for col in numeric_cols:
    lower, upper = iqr_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {"lower_bound": round(lower, 2), "upper_bound": round(upper, 2),
                            "n_outliers": n_outliers, "pct_outliers": round(n_outliers / len(df) * 100, 2)}

pd.DataFrame(outlier_report).T

,lower_bound,upper_bound,n_outliers,pct_outliers
ticket_price,-85.00,235.00,2888.0,2.89
spend_amount,-356.64,855.72,27.0,0.03


In [ ]:
df_clean = df.copy()
for col in numeric_cols:
    lower, upper = iqr_bounds(df[col])
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

print("Before capping - spend_amount max:", df["spend_amount"].max())
print("After capping  - spend_amount max:", df_clean["spend_amount"].max())

Before capping - spend_amount max: 898.22
After capping  - spend_amount max: 855.7249999999999


## Save Output for the Next Teammate
This notebook is **3rd** in the run order. Save the outlier-capped dataset for the next person (Encoding).

In [5]:
df_clean.to_csv("output_after_outlier_removal.csv", index=False)
print("Saved output_after_outlier_removal.csv - shape:", df_clean.shape)


Saved output_after_outlier_removal.csv - shape: (100000, 6)
